In [3]:
import json
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import silhouette_score
from IPython.display import HTML


In [4]:
# DATA DIRECTORIES
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'newyorker_caption_contest'
JSON_DIR = DATA_DIR / 'contests.json'
CSV_DIR = DATA_DIR / 'data'
CSV_CLEAN_DIR = DATA_DIR / 'data_cleaned'
IMG_DIR = DATA_DIR / 'images'
DOC_THEMES_DIR = DATA_DIR / 'THEMES_RESULTS_V4'/'docs_with_themes.csv'
THEMES_TFIDF_DIR=DATA_DIR/'THEMES_RESULTS_V4'/'TF_IDF_themes_scores.csv'
THEMES_DIR=DATA_DIR/'THEMES_RESULTS_V4'/'themes.csv'




## 0. Cleaning the data set with spacy

We used spacy to clean the raw data set to have more meaningfull themes

In [48]:

docs = []

for csv_path in sorted(CSV_CLEAN_DIR.glob("*.csv")):
    df = pd.read_csv(csv_path)
    contest_id = csv_path.stem

    for idx, row in df.iterrows():
        text = str(row.get("caption", "")).strip()
        if not text:
            continue

        docs.append({
            "contest_id": contest_id,
            "caption_id": idx,
            "text": text,

            # metadata (if present)
            "rank": row.get("rank"),
            "mean": row.get("mean"),
            "precision": row.get("precision"),
            "votes": row.get("votes"),
            "funny": row.get("funny"),
            "somewhat_funny": row.get("somewhat_funny"),
            "not_funny": row.get("not_funny"),
        })

docs_df = pd.DataFrame(docs)
print("docs_df shape:", docs_df.shape)


docs_df shape: (2263088, 10)


In [49]:
docs_df.head()

,contest_id,caption_id,text,rank,mean,precision,votes,funny,somewhat_funny,not_funny
0,510_cleaned,0,I'm a congressman--obstruction is my job.,0,1.913043,0.094022,69,18,27,24
1,510_cleaned,1,"I'm what they mean when they say, 'The middle ...",1,1.842105,0.191381,19,5,6,8
2,510_cleaned,2,Does this suit make me look flat?,2,1.711111,0.112915,45,8,16,21
3,510_cleaned,3,"When the right woman comes along, I'll know it.",3,1.625000,0.116657,32,3,14,15
4,510_cleaned,4,"I used to lie in the gutter, but then I quit d...",4,1.617647,0.133610,34,6,9,19


In [51]:
custom_stopwords = {
    "like", "don", "ll", "kinda", "sorta",
}

for w in custom_stopwords:
    nlp.Defaults.stop_words.add(w)

In [54]:
import spacy
from tqdm import tqdm

# Charger spaCy
nlp = spacy.load("en_core_web_sm")

# Activer tqdm pour pandas
tqdm.pandas()

def clean_caption(text):
    """
    Clean a caption by:
    - running spaCy
    - removing stop words and non-content words
    - keeping only alphabetic NOUN/PROPN/ADJ/VERB
    - lemmatizing and lowercasing
    """
    doc = nlp(str(text))  # str() in case of NaN
    tokens = []
    for token in doc:
        if (
            token.is_alpha
            and not token.is_stop
            and token.pos_ in {"NOUN", "PROPN", "ADJ", "VERB"}
        ):
            tokens.append(token.lemma_.lower())
    return " ".join(tokens)

# Apply with progress bar
docs_df["clean_text"] = docs_df["text"].progress_apply(clean_caption)


100%|██████████| 2263088/2263088 [1:32:53<00:00, 406.03it/s]


In [55]:
docs_df.keys()

Index(['contest_id', 'caption_id', 'text', 'rank', 'mean', 'precision',
       'votes', 'funny', 'somewhat_funny', 'not_funny', 'clean_text'],
      dtype='object')

In [56]:
# cols_export = ['contest_id', 'caption_id', 'text', 'clean_text', 'rank', 'mean',
#                'precision', 'votes', 'funny', 'somewhat_funny', 'not_funny']

# export csv
docs_df.to_csv("/Users/mariem/Downloads/docs_clean_captions.csv", index=False)

In [50]:
import spacy
nlp = spacy.load("en_core_web_sm")

def clean_caption(text):
    doc = nlp(str(text))
    toks = [
        t.lemma_.lower()
        for t in doc
        if t.is_alpha and not t.is_stop and t.pos_ in {"NOUN","PROPN","ADJ","VERB"}
    ]
    return " ".join(toks)

docs_df["clean_text"] = docs_df["text"].apply(clean_caption)

KeyboardInterrupt: 

In [29]:
# Count captions per theme
theme_freq = (
    docs_df["theme_id"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "theme_id", "theme_id": "n_captions"})
)

# Show top 10 most frequent themes
theme_freq.head(10)



,n_captions,count
0,75,100287
1,42,97886
2,35,53470
3,8,48931
4,57,47142
5,44,47141
6,39,45188
7,73,44259
8,32,43654
9,13,43395


Check if recurrent words in our top themes are in the library

In [ ]:

words_to_check = ["like", "don't", "don", "just", "i", "ll", "im", "kinda", "sorta"]

for w in words_to_check:
    doc = nlp(w)
    token = doc[0]
    print(f"{w:10} -> is_stop={token.is_stop}, lemma={token.lemma_}")


like       -> is_stop=True, lemma=like
don't      -> is_stop=True, lemma=do
don        -> is_stop=True, lemma=don
just       -> is_stop=True, lemma=just
i          -> is_stop=True, lemma=I
ll         -> is_stop=True, lemma=ll
im         -> is_stop=True, lemma=I
kinda      -> is_stop=True, lemma=kinda
sorta      -> is_stop=True, lemma=sorta


So It seems we have to add a few stop words 


In [8]:
custom_stopwords = {
    "like", "don", "ll", "kinda", "sorta",
}

for w in custom_stopwords:
    nlp.Defaults.stop_words.add(w)


In [ ]:

def clean_caption(text):
    """
    Clean a caption by:
    - running spaCy
    - removing stop words and non-content words
    - keeping only alphabetic NOUN/PROPN/ADJ/VERB
    - lemmatizing and lowercasing
    """
    doc = nlp(str(text))  # str() in case of NaN
    tokens = []
    for token in doc:
        if (
            token.is_alpha
            and not token.is_stop
            and token.pos_ in {"NOUN", "PROPN", "ADJ", "VERB"}
        ):
            tokens.append(token.lemma_.lower())
    return " ".join(tokens)
docs_df["clean_text"] = docs_df["text"].apply(clean_caption)




Now we applied Bert, clustering then TF-IDF on the cleaned text and we obtained the different themes.

## 0. Let's understand our results:


In order to further analyze our results, we need to understand what each column represents. Here is an overview of both the themes.csv and TF_IDF_themes_scores.csv in more detail.

In [5]:

themes_dfidf = pd.read_csv(THEMES_TFIDF_DIR)
themes_dfidf.head(10)

,theme_id,term,sim,local_freq,global_freq,distinctiveness,score
0,0,dog,0.791499,14256,18522,0.769679,0.609201
1,0,puppy,0.721091,748,928,0.806034,0.581225
2,0,pet,0.747876,2941,4706,0.624947,0.467383
3,0,bark,0.658491,1355,2078,0.652069,0.429382
4,0,leash,0.576722,982,1334,0.736132,0.424544
5,0,vet,0.611917,628,1344,0.467262,0.285925
6,0,collar,0.622866,516,1370,0.376642,0.234598
7,0,animal,0.631270,936,4334,0.215967,0.136333
8,0,know,0.257657,1311,84565,0.015503,0.003994
9,0,think,0.233868,1379,131386,0.010496,0.002455


-Local_freq: Number of documents inside this theme where the term appears at least once. High → the term is representative of this theme
Low → the term is marginal or accidental

-global_freq: Number of documents in the entire corpus where the term appears. Low → more specific / niche

-distinctiveness: How unique the term is to this theme.≈ 1 → almost exclusive to this theme

-score:The final ranking score for a term as a theme label.A good label must be: Meaningful → high sim AND Specific → high distinctivenes

In [6]:
#visualize the theme csv
themes_df = pd.read_csv(THEMES_DIR)
themes_df.head(10)

,theme_id,theme_label,nb_instances,mu_rank,mu_mean,mu_precision,mu_not_funny,mu_somewhat_funny,mu_funny,TF_IDF_terms
0,0,Dog,28929,3221.119071,1.211799,0.047786,100.960863,27.160904,10.610151,"dog, pet, bark, leash, puppy, animal, vet, kno..."
1,1,Fall Asleep,18290,3005.333206,1.217083,0.054425,93.689978,27.131281,11.584067,"sleep, night, tired, dream, bed, wake, nap, as..."
2,2,Year,21754,3161.643560,1.226651,0.051957,98.822745,30.887515,13.722948,"year, month, season, fall, date, time, age, th..."
3,3,Work Home,35262,3162.187975,1.218767,0.055696,88.715088,25.710210,10.443817,"work, job, work home, home, office, say, think..."
4,4,Clown,22401,3421.341284,1.205269,0.051263,88.639421,21.413864,8.350485,"clown, circus, laugh, funny, cop, joke, tell, ..."
5,5,Elephant Room,17598,3431.416596,1.214021,0.054945,87.008303,23.395439,9.781778,"room, elephant, elephant room, door, talk, spa..."
6,6,Vegan,17207,3646.708316,1.198785,0.055109,84.196315,18.852676,6.405184,"vegan, gluten, gluten free, free, vegetarian, ..."
7,7,Hell,16298,3134.965885,1.219931,0.056697,93.928703,26.945944,11.085716,"think, way, need, know, say, hell, look, thing..."
8,8,Boy,66343,3121.448548,1.213524,0.053852,92.028956,24.131830,9.542740,"good, way, right, bad, thank, thing, guy, nice..."
9,9,Bird,30228,3180.440007,1.191049,0.051456,94.127465,23.745897,9.198478,"bird, fly, wing, pigeon, duck, chicken, egg, f..."


-nb_instances: Number of captions assigned to this theme.

-mu_rank: Average rank value of captions inside the theme.

-mu_mean: Mean of the mean score across captions in the theme. High → captions in this theme perform well on average

-mu_precision:Average precision across captions in the theme.High → theme performs well consistently

-mu_not_funny:Average proportion of “not funny” judgments for captions in this theme.



## 1. Visualizing theme frequencies:


We used the Flourish Hierarchy Circles chart type to visualize the frequencies of each global Theme and it's 10 disctinctive terms using a new metric that we will call Size. this new metric uses the score value of the TF_IDF_themes_scores.csv and the nb_instances value of the themes.csv. The size of each term bubble is computed by normalizing the TF-IDF score of the term within its theme and scaling it by the number of instances in that theme, so that the total area of a theme reflects its prevalence in the dataset while the relative sizes of the nested terms indicate their semantic importance.


Formally, term size is defined as:
  
$\text{Size}_{t,c} = \frac{\text{TF-IDF}_{t,c}}{\sum_{t'} \text{TF-IDF}_{t',c}} \times nb-instances$,  
where $nb-instances$ is the number of documents in theme $c$.


Now we create the csv file that we will upload on flourish to get the chart.

In [14]:
# 1. Merge TF-IDF terms with theme metadata
df = (
    themes_dfidf
    .merge(
        themes_df[["theme_id", "theme_label", "nb_instances"]],
        on="theme_id",
        how="left"
    )
)

# 2. Compute sum of term scores per theme
df["sum_term_score_in_theme"] = (
    df.groupby("theme_id")["score"].transform("sum")
)

# 3. Compute Size
df["Size"] = (
    df["score"] / df["sum_term_score_in_theme"]
) * df["nb_instances"]

df_final = df[["term", "theme_label", "Size"]]
df_final.head(10)

df_final.to_csv("/Users/mariem/Desktop/Project_ADA/theme_visualization_f8.csv", index=False)

In [13]:

# Display the Flourish Sankey diagram (with fixed width)
HTML("""<div class="flourish-embed flourish-hierarchy" data-src="visualisation/26906609"><script src="https://public.flourish.studio/resources/embed.js"></script><noscript><img src="https://public.flourish.studio/visualisation/26906609/thumbnail" width="100%" alt="hierarchy visualization" /></noscript></div>
""")

## 3. Generate Theme labels with LLM

This code uses a large language model (via the Ollama API) to automatically generate a human-readable label for each theme based on its TF-IDF terms. For every row in `themes_df`, the TF-IDF keywords describing the theme are sent as a prompt to the model, which is asked to return a single representative theme label in a fixed format.


In [ ]:
import ollama
import os
from tqdm import tqdm

os.environ['OLLAMA_API_KEY'] = 'f4022fdc9ef54ce7b5c3d6f48a6e0a68.99acHyAHtoF9C-e2QuiCFPYz'
client = ollama.Client(
        host='https://ollama.com',
        headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')} # this is my key 
        )

results = {}
for id, row in tqdm(themes_df.iterrows()):
    tf_idf=row['TF_IDF_terms']
    prompt=f"Here are the TF-IDF terms for theme {id} : {tf_idf}. Please generate a word representative of the 10 tf-idf terms \n"
    prompt+=f"please your answer must be as following : id=... , LLMS_theme=..."
    messages = [{
                    'role': 'user',
                    'content': prompt,
                },]
    full_response = ''
    # Stream the response and accumulate it
    for part in client.chat('gpt-oss:120b', messages=messages, stream=True): #
        if part.message.content:
            content = part.message.content
            full_response += content
    results[str(id)] = full_response


def store_joke_classification(classifications):
    rows = []
    for cap_id, text in classifications.items():
        
        # Extract numbers and labels
        parts = [p.strip() for p in text.split(',')][1]
        
        print(parts)
        label = parts.removeprefix('LLMS_theme=')
        
        # build row
        row = {
            "caption_id": cap_id,
            "gen_label":label}
        rows.append(row)
        print(row)
    print(rows)
    pd.DataFrame(rows).to_csv('/Users/mariem/Downloads/Themes_augmented_3.csv',index=False)

store_joke_classification(results)


80it [02:03,  1.55s/it]

LLMS_theme=dog
{'caption_id': '0', 'gen_label': 'dog'}
LLMS_theme=sleep
{'caption_id': '1', 'gen_label': 'sleep'}
LLMS_theme=time
{'caption_id': '2', 'gen_label': 'time'}
LLMS_theme=work
{'caption_id': '3', 'gen_label': 'work'}
LLMS_theme=comedy
{'caption_id': '4', 'gen_label': 'comedy'}
LLMS_theme=acknowledgment
{'caption_id': '5', 'gen_label': 'acknowledgment'}
LLMS_theme=dietary
{'caption_id': '6', 'gen_label': 'dietary'}
LLMS_theme=understanding
{'caption_id': '7', 'gen_label': 'understanding'}
LLMS_theme=goodness
{'caption_id': '8', 'gen_label': 'goodness'}
LLMS_theme=bird
{'caption_id': '9', 'gen_label': 'bird'}
LLMS_theme=hair
{'caption_id': '10', 'gen_label': 'hair'}
LLMS_theme=size
{'caption_id': '11', 'gen_label': 'size'}
LLMS_theme=meaning
{'caption_id': '12', 'gen_label': 'meaning'}
LLMS_theme=unknown
{'caption_id': '13', 'gen_label': 'unknown'}
LLMS_theme=technology
{'caption_id': '14', 'gen_label': 'technology'}
LLMS_theme=transition
{'caption_id': '15', 'gen_label': 'tra

Now lets visualize the theme frequencies again with the new generated themes.

In [115]:
Themes_aug_DIR = DATA_DIR / 'Themes_augmented_3.csv'
themes_df = pd.read_csv(THEMES_DIR)
llm_df = pd.read_csv(Themes_aug_DIR)
tfidf_df = pd.read_csv(THEMES_TFIDF_DIR)

In [ ]:
theme_id_to_llm = dict(
    zip(llm_df["caption_id"].astype(int), llm_df["gen_label"])
)

themes_df["theme_label_llm"] = themes_df["theme_id"].map(theme_id_to_llm)
themes_df["theme_label_llm"] = themes_df["theme_label_llm"].fillna(
    themes_df["theme_label"]
).str.title()
# --------------------------------------------------

df = tfidf_df.merge(
    themes_df[["theme_id", "theme_label_llm", "nb_instances"]],
    on="theme_id",
    how="left"
)

# --------------------------------------------------
# Recompute Size
# Size = (term_score / sum(term_scores_in_theme)) * nb_instances
# --------------------------------------------------
df["sum_term_score_in_theme"] = (
    df.groupby("theme_id")["score"].transform("sum")
)

df["Size"] = (
    df["score"] / df["sum_term_score_in_theme"]
) * df["nb_instances"]

# --------------------------------------------------
# Final Flourish hierarchy CSV
# --------------------------------------------------
flourish_hierarchy_llm = df[
    ["theme_label_llm", "term", "Size"]
].rename(columns={"theme_label_llm": "theme_label"})

# --------------------------------------------------
# Export
# --------------------------------------------------
out_path = "/Users/mariem/Downloads/flourish_hierarchy_llm_labels.csv"
flourish_hierarchy_llm.to_csv(out_path, index=False)

print(f"Flourish hierarchy CSV saved to: {out_path}")


Flourish hierarchy CSV saved to: /Users/mariem/Downloads/flourish_hierarchy_llm_labels.csv


In [120]:
# Display the Flourish Sankey diagram (with fixed width)
HTML("""
<div class="flourish-embed flourish-hierarchy" data-src="visualisation/26902472"><script src="https://public.flourish.studio/resources/embed.js"></script><noscript><img src="https://public.flourish.studio/visualisation/26902472/thumbnail" width="100%" alt="hierarchy visualization" /></noscript></div>
""")